This notebook covers the managed file storage layer of AWS: EFS for Linux shared filesystems, the FSx family for specialised workloads (Windows, HPC, NetApp), and Storage Gateway for connecting on-premises environments to AWS storage. Together these services cover every file storage scenario you'll encounter at the Solutions Architect level.

## Amazon EFS — Elastic File System

EFS is a fully managed **NFS (Network File System)** that can be mounted on thousands of EC2 instances simultaneously — across multiple AZs, across accounts, and even from on-premises via Direct Connect or VPN.

### Key characteristics
- **Protocol**: NFSv4.1 / NFSv4.2
- **OS**: Linux only (POSIX-compliant filesystem)
- **Scope**: Regional — data replicated across multiple AZs automatically
- **Capacity**: Elastic — grows and shrinks automatically, no provisioning needed
- **Pricing**: Pay per GB stored (no upfront provisioning)

### How mounting works
You create **mount targets** — one per AZ you want to access EFS from. Each mount target gets a private IP address in your VPC subnet. EC2 instances mount EFS via that IP (or the EFS DNS name). Security groups on the mount target control which instances can connect.

```
EFS Filesystem
    ├── Mount Target  us-east-1a  (10.0.1.5)  ← instances in AZ-a mount here
    ├── Mount Target  us-east-1b  (10.0.2.5)  ← instances in AZ-b mount here
    └── Mount Target  us-east-1c  (10.0.3.5)  ← instances in AZ-c mount here
```

All mount targets point to the same filesystem — data written via any mount target is immediately visible from all others.

### Performance modes

| Mode | Use case |
|---|---|
| **General Purpose** (default) | Latency-sensitive workloads — web serving, CMS, home directories |
| **Max I/O** | Highly parallelised workloads with many thousands of concurrent operations — big data, media processing. Higher latency. |

### Throughput modes

| Mode | How throughput is determined |
|---|---|
| **Elastic** (recommended) | Automatically scales throughput up and down with workload. Pay per GB transferred. Best for spiky or unpredictable workloads. |
| **Bursting** | Baseline throughput scales with storage size (50 KB/s per GB). Earn burst credits when below baseline, spend them on bursts. |
| **Provisioned** | Set a fixed throughput independent of storage size. Use when you need consistent high throughput but have a small dataset. |

### Storage tiers and lifecycle

| Tier | Use case | Cost |
|---|---|---|
| **Standard** | Frequently accessed files | Highest |
| **Standard-IA** | Infrequently accessed (lifecycle policy moves files after N days) | ~92% cheaper than Standard |
| **Archive** | Rarely accessed (lifecycle policy) | ~95% cheaper than Standard |
| **One Zone** | Single-AZ variant of Standard — 47% cheaper; for non-critical data | Lower |
| **One Zone-IA** | Single-AZ IA | Lowest |

EFS lifecycle management automatically moves files between tiers based on last-access time (e.g., move to IA after 30 days of no access).

### Common use cases
- Content management systems (WordPress, Drupal) shared across multiple web servers
- Home directories for thousands of Linux users
- Shared code repositories or configuration files across an Auto Scaling Group
- Container storage in ECS/EKS (EFS CSI driver)
- Machine learning training data shared across many GPU instances

> **Exam tip:** EFS = Linux + multi-AZ + shared across many instances. Windows = FSx for Windows File Server.

## Amazon FSx

FSx is a family of fully managed third-party filesystems. Each variant is purpose-built for specific workloads:

| Variant | Protocol | Use case |
|---|---|---|
| **FSx for Windows File Server** | SMB | Windows workloads needing NTFS and Active Directory |
| **FSx for Lustre** | Lustre (POSIX) | High-performance computing, ML, big data |
| **FSx for NetApp ONTAP** | NFS, SMB, iSCSI | Enterprise multi-protocol, lift-and-shift from NetApp |
| **FSx for OpenZFS** | NFS | ZFS-based workloads, fast clones and snapshots |

### FSx for Windows File Server

A fully managed Windows filesystem backed by Windows Server. It supports:
- **SMB protocol** (Server Message Block) and **NTFS** — Windows-native
- **Active Directory integration** — join to your corporate AD (self-managed or AWS Managed Microsoft AD)
- **DFS Namespaces** — distribute files across multiple FSx filesystems under a single namespace
- **Multi-AZ deployment** — automatic failover to a standby in another AZ
- Daily automated backups to S3

**Use case**: Lift-and-shift Windows applications that rely on SMB shares, SQL Server workloads that use a shared filesystem, user home directories for Windows desktops.

> **Exam tip:** If the scenario mentions Windows, SMB, NTFS, or Active Directory file shares — FSx for Windows File Server, not EFS.

### FSx for Lustre

Lustre is a high-performance parallel filesystem used in HPC and ML. FSx for Lustre delivers:
- Up to **hundreds of GB/s** throughput and millions of IOPS
- Sub-millisecond latency
- **S3 integration**: link an FSx for Lustre filesystem to an S3 bucket — files appear in the filesystem lazily loaded on first access; processed results can be exported back to S3

### Deployment types

| Type | Durability | Use case |
|---|---|---|
| **Scratch** (1 or 2) | Data not replicated — temporary | Short-term processing, cost optimisation |
| **Persistent** | Replicated within AZ | Long-running jobs, sensitive data |

**Use cases**: Machine learning training, genomics research, financial simulations, video processing, seismic analysis — workloads that process data fast and at scale.

> **Exam tip:** HPC, ML training at scale, or "high throughput filesystem" → FSx for Lustre. The S3 integration (read data from S3, process, write back) is a common exam scenario.

### FSx for NetApp ONTAP

Managed NetApp ONTAP filesystem on AWS. Supports **NFS, SMB, and iSCSI** simultaneously — making it useful for mixed workloads (Linux and Windows clients sharing the same filesystem).

Key features:
- **SnapMirror** — replicate data to other ONTAP systems (on-premises or other AWS regions)
- **Instant clones** — space-efficient volume clones for dev/test
- **Automatic tiering** — automatically moves cold data to lower-cost S3 storage
- **Shrink volumes** — unlike EFS, volumes can shrink as well as grow

**Use case**: Organisations already running NetApp on-premises who want to lift-and-shift without retraining staff or rearchitecting applications.

### FSx for OpenZFS

Managed OpenZFS filesystem. Key features:
- Up to 1 million IOPS with sub-millisecond latency
- **Instant snapshots** and **zero-cost clones** (point-in-time copy with copy-on-write)
- NFS protocol (Linux and macOS compatible)
- Point-in-time recovery

**Use case**: Lift-and-shift from on-premises ZFS; dev/test environments that need fast cloning of large datasets.

## AWS Storage Gateway

Storage Gateway bridges on-premises environments to AWS storage. It runs as a VM appliance (or hardware appliance) in your data centre and presents AWS storage as local storage to your on-premises systems.

There are four gateway types:

### S3 File Gateway

Presents an S3 bucket as a **NFS or SMB file share** to on-premises clients. Files written through the gateway are stored as native S3 objects — accessible directly via the S3 API as well as through the file share.

- Local cache for recently accessed files (low latency for hot data)
- Supports all S3 storage classes including S3-IA and Glacier via lifecycle policies
- Ideal for: file shares that need to land in S3, backup targets, archiving on-premises file data to S3

```
On-premises app → NFS/SMB → S3 File Gateway → S3 bucket
                                    ↑ local cache for hot files
```

### FSx File Gateway

Local cache in front of **FSx for Windows File Server**. On-premises Windows clients access the gateway via SMB; frequently accessed files are cached locally for low latency.

- Reduces WAN latency for remote office users accessing centralised FSx
- Full AD integration

### Volume Gateway

Presents iSCSI block volumes to on-premises servers. Two modes:

| Mode | Primary storage | Backup | Use case |
|---|---|---|---|
| **Stored** | On-premises (full dataset locally) | Async snapshots to S3 as EBS snapshots | Low-latency access to full dataset + cloud backup |
| **Cached** | S3 (primary) | — | Only recently accessed data cached on-premises; full dataset in S3 |

- Snapshots stored as EBS snapshots → can be restored as EBS volumes directly
- Stored mode: entire dataset on-premises, async backup to AWS
- Cached mode: only cache on-premises, primary data in AWS — minimises on-premises storage

### Tape Gateway

Virtual Tape Library (VTL) that integrates with existing **backup software** (Veeam, Veritas, Commvault). On-premises backup software sees virtual tape drives and shelves. Tapes are stored in S3 and can be archived to S3 Glacier or Glacier Deep Archive.

- Replaces physical tape infrastructure without changing backup workflows
- Archived virtual tapes go to Glacier automatically

### Storage Gateway summary

| Gateway | Protocol | Primary storage | Use case |
|---|---|---|---|
| S3 File | NFS / SMB | S3 | On-prem file shares → S3, archiving |
| FSx File | SMB | FSx for Windows | Remote office cache for FSx |
| Volume — Stored | iSCSI | On-premises | Full local data + async cloud backup |
| Volume — Cached | iSCSI | S3 | Minimal on-premises storage, primary in S3 |
| Tape | iSCSI VTL | S3 / Glacier | Replace physical tape drives |

## Working with EFS and FSx via boto3

In [ ]:
import boto3

efs = boto3.client('efs', region_name='us-east-1')
fsx = boto3.client('fsx', region_name='us-east-1')

In [ ]:
# Create an EFS filesystem with Elastic throughput and lifecycle management
import uuid

fs = efs.create_file_system(
    CreationToken=str(uuid.uuid4()),
    PerformanceMode='generalPurpose',
    ThroughputMode='elastic',       # recommended: auto-scales with workload
    Encrypted=True,
    Tags=[{'Key': 'Name', 'Value': 'app-shared-fs'}]
)
fs_id = fs['FileSystemId']
print(f"Filesystem: {fs_id}")

# Set lifecycle: move to IA after 30 days, to Archive after 90 days
efs.put_lifecycle_configuration(
    FileSystemId=fs_id,
    LifecyclePolicies=[
        {'TransitionToIA': 'AFTER_30_DAYS'},
        {'TransitionToArchive': 'AFTER_90_DAYS'},
        {'TransitionToPrimaryStorageClass': 'AFTER_1_ACCESS'}  # promote on read
    ]
)
print("Lifecycle configured: Standard → IA (30d) → Archive (90d), promote on access")

In [ ]:
# Create mount targets in two AZs
# Each mount target needs a subnet and security group
subnet_az_a = 'subnet-0abc111'
subnet_az_b = 'subnet-0abc222'
sg_id       = 'sg-0efs123'

for subnet in [subnet_az_a, subnet_az_b]:
    mt = efs.create_mount_target(
        FileSystemId=fs_id,
        SubnetId=subnet,
        SecurityGroups=[sg_id]
    )
    print(f"Mount target {mt['MountTargetId']} in {mt['AvailabilityZoneName']} — {mt['IpAddress']}")

In [ ]:
# EFS Access Points — scoped entry points with enforced UID/GID and root directory
# Useful for containers: each pod gets its own access point with isolated directory
ap = efs.create_access_point(
    FileSystemId=fs_id,
    PosixUser={'Uid': 1000, 'Gid': 1000},
    RootDirectory={
        'Path': '/app-data',
        'CreationInfo': {
            'OwnerUid': 1000,
            'OwnerGid': 1000,
            'Permissions': '755'
        }
    },
    Tags=[{'Key': 'Name', 'Value': 'app-access-point'}]
)
print(f"Access point: {ap['AccessPointId']}")

In [ ]:
# Create FSx for Windows File Server (Multi-AZ)
fsx_windows = fsx.create_file_system(
    FileSystemType='WINDOWS',
    StorageCapacity=300,    # GB
    StorageType='SSD',
    SubnetIds=['subnet-0abc111', 'subnet-0abc222'],  # 2 subnets = Multi-AZ
    SecurityGroupIds=['sg-0fsx456'],
    WindowsConfiguration={
        'ActiveDirectoryId': 'd-1234567890',   # AWS Managed Microsoft AD
        'ThroughputCapacity': 32,              # MB/s
        'DeploymentType': 'MULTI_AZ_1',        # automatic failover
        'PreferredSubnetId': 'subnet-0abc111',
        'AutomaticBackupRetentionDays': 7,
        'CopyTagsToBackups': True
    },
    Tags=[{'Key': 'Name', 'Value': 'corp-file-share'}]
)
print(f"FSx for Windows: {fsx_windows['FileSystem']['FileSystemId']}")

In [ ]:
# Create FSx for Lustre linked to S3 (for ML training data)
fsx_lustre = fsx.create_file_system(
    FileSystemType='LUSTRE',
    StorageCapacity=1200,   # GB — must be multiple of 1200 for SCRATCH_2
    StorageType='SSD',
    SubnetIds=['subnet-0abc111'],
    SecurityGroupIds=['sg-0lustre789'],
    LustreConfiguration={
        'ImportPath': 's3://ml-training-data/',        # read data from S3
        'ExportPath': 's3://ml-training-data/results', # write results back
        'DeploymentType': 'SCRATCH_2',  # fast, temporary — no replication
        'AutoImportPolicy': 'NEW_CHANGED_DELETED'  # sync S3 changes to filesystem
    },
    Tags=[{'Key': 'Name', 'Value': 'ml-training-scratch'}]
)
print(f"FSx for Lustre: {fsx_lustre['FileSystem']['FileSystemId']}")

## Decision Guide: Which File Storage to Use?

```
Need shared file storage?
│
├── Linux instances, POSIX, multi-AZ shared?
│       → EFS
│
├── Windows, SMB, Active Directory?
│       → FSx for Windows File Server
│
├── HPC, ML training, maximum throughput?
│       → FSx for Lustre
│           └── Data lives in S3? → Link Lustre to S3 bucket
│
├── Existing NetApp ONTAP, multi-protocol (NFS+SMB+iSCSI)?
│       → FSx for NetApp ONTAP
│
├── Existing ZFS, fast clones needed?
│       → FSx for OpenZFS
│
└── On-premises connecting to AWS storage?
        ├── Files → S3:           S3 File Gateway
        ├── Files → FSx Windows:  FSx File Gateway
        ├── Block + local copy:   Volume Gateway (Stored)
        ├── Block + cloud primary: Volume Gateway (Cached)
        └── Replace tape drives:  Tape Gateway
```

## Summary

| Concept | Key Takeaway |
|---|---|
| EFS | Managed NFS, Linux only, multi-AZ, scales automatically, thousands of concurrent mounts |
| EFS throughput: Elastic | Auto-scales with workload — recommended default |
| EFS throughput: Provisioned | Fixed throughput for small dataset with high throughput needs |
| EFS One Zone | Single-AZ variant, 47% cheaper, for non-critical data |
| EFS lifecycle | Automatically moves files to IA/Archive based on last-access time |
| FSx for Windows | SMB + NTFS + Active Directory — for Windows workloads |
| FSx for Lustre | HPC/ML — highest throughput, S3 integration for training data |
| FSx Lustre Scratch | Temporary, no replication — short processing jobs |
| FSx Lustre Persistent | Replicated within AZ — long-running jobs |
| FSx for NetApp ONTAP | Multi-protocol (NFS/SMB/iSCSI), lift-and-shift from NetApp |
| FSx for OpenZFS | Fast clones and snapshots, lift-and-shift from ZFS |
| S3 File Gateway | On-prem NFS/SMB share backed by S3 — files stored as S3 objects |
| FSx File Gateway | Local SMB cache for FSx for Windows — remote office latency |
| Volume Gateway Stored | Full dataset on-prem, async backup snapshots to S3 |
| Volume Gateway Cached | Primary data in S3, only hot data cached on-prem |
| Tape Gateway | Virtual tape library — replace physical tapes with S3/Glacier |